In [1]:
import pandas as pd
import time
import sys
import os


root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.append(root_path)

from python.geo_location import haversine_distance

In [ ]:
bigfoot_df = pd.read_csv("../data/processed/combined_bigfoot_v1.csv")
uap_df = pd.read_csv("../data/processed/us_uap_1940_v1.csv")

In [196]:
proximity_df = pd.merge(
    bigfoot_df[[
        'bf_id', 'latitude', 'longitude', 'full_date',
        'geohash_7', 'geohash_6', 'geohash_5', 'year'
    ]],
    uap_df[[
        'uap_id', 'full_date', 'city', 'state_code',
        'shape_group', 'geohash_7', 'geohash_6', 'geohash_5', 'latitude', 'longitude', 'year'
    ]],
    on='geohash_6',
    how='inner',
    suffixes=('_bf', '_uap')
)


In [197]:
proximity_df['full_date_bf'] = pd.to_datetime(proximity_df['full_date_bf'], format="%Y-%m-%d", errors="coerce")
proximity_df['full_date_uap'] = pd.to_datetime(proximity_df['full_date_uap'], format="%Y-%m-%d", errors="coerce")

In [198]:
# date difference in days

proximity_df['date_diff_days'] = (
    proximity_df['full_date_bf'] - proximity_df['full_date_uap']
).dt.days.abs()

In [199]:
proximity_df = proximity_df[[
    'bf_id', 'uap_id',
    'full_date_bf', 'full_date_uap', 'date_diff_days',
    'geohash_6', 'geohash_7_bf', 'geohash_7_uap',
    'city', 'state_code',
    'year_bf', 'year_uap',
    'latitude_bf', 'longitude_bf', 'latitude_uap', 'longitude_uap'
]]

In [200]:
proximity_df.head()

,bf_id,uap_id,full_date_bf,full_date_uap,date_diff_days,geohash_6,geohash_7_bf,geohash_7_uap,city,state_code,year_bf,year_uap,latitude_bf,longitude_bf,latitude_uap,longitude_uap
0,147,2668,2000-06-15,1976-05-29,8783,9qftmx,9qftmxw,9qftmxw,south lake tahoe,CA,2000,1976,38.93333,-119.9833,38.933333,-119.983333
1,147,3657,2000-06-15,1980-06-30,7290,9qftmx,9qftmxw,9qftmxw,south lake tahoe,CA,2000,1980,38.93333,-119.9833,38.933333,-119.983333
2,147,10777,2000-06-15,1998-11-17,576,9qftmx,9qftmxw,9qftmxw,south lake tahoe (meyers),CA,2000,1998,38.93333,-119.9833,38.933333,-119.983333
3,147,18147,2000-06-15,2001-11-12,515,9qftmx,9qftmxw,9qftmxw,south lake tahoe,CA,2000,2001,38.93333,-119.9833,38.933333,-119.983333
4,147,25217,2000-06-15,2004-04-11,1396,9qftmx,9qftmxw,9qftmxw,south lake tahoe,CA,2000,2004,38.93333,-119.9833,38.933333,-119.983333


In [201]:
proximity_df['distance_meters'] = round (1000 * haversine_distance(
    proximity_df['latitude_bf'],
    proximity_df['longitude_bf'],
    proximity_df['latitude_uap'],
    proximity_df['longitude_uap']
))

# verify (descending, since many values are 0)
proximity_df.sort_values('distance_meters', ascending=False).head(50)

,bf_id,uap_id,full_date_bf,full_date_uap,date_diff_days,geohash_6,geohash_7_bf,geohash_7_uap,city,state_code,year_bf,year_uap,latitude_bf,longitude_bf,latitude_uap,longitude_uap,distance_meters
206,10919,48867,1975-09-15,2010-08-27,12765,9ynqrg,9ynqrgb,9ynqrgp,jacksonville,AR,1975,2010,34.87000,-92.12000,34.866111,-92.110000,1010.0
205,10919,21305,1975-09-15,2003-01-18,9987,9ynqrg,9ynqrgb,9ynqrgp,jacksonville,AR,1975,2003,34.87000,-92.12000,34.866111,-92.110000,1010.0
204,10919,2901,1975-09-15,1977-05-01,594,9ynqrg,9ynqrgb,9ynqrgp,jacksonville,AR,1975,1977,34.87000,-92.12000,34.866111,-92.110000,1010.0
10,502,3859,2000-06-17,1981-07-28,6899,9qfjfx,9qfjfx0,9qfjfxr,foresthill,CA,2000,1981,39.01917,-120.82750,39.020278,-120.816944,920.0
11,502,34467,2000-06-17,2006-12-31,2388,9qfjfx,9qfjfx0,9qfjfxr,foresthill,CA,2000,2006,39.01917,-120.82750,39.020278,-120.816944,920.0
12,502,57459,2000-06-17,2012-06-21,4387,9qfjfx,9qfjfx0,9qfjfxr,foresthill,CA,2000,2012,39.01917,-120.82750,39.020278,-120.816944,920.0
242,25592,41765,1986-05-24,2008-10-27,8192,dnd2p7,dnd2p7j,dnd2p72,portland,TN,1986,2008,36.58020,-86.51040,36.581667,-86.516389,559.0
243,25592,47901,1986-05-24,2010-07-04,8807,dnd2p7,dnd2p7j,dnd2p72,portland,TN,1986,2010,36.58020,-86.51040,36.581667,-86.516389,559.0
244,25592,52097,1986-05-24,2011-07-04,9172,dnd2p7,dnd2p7j,dnd2p72,portland,TN,1986,2011,36.58020,-86.51040,36.581667,-86.516389,559.0
245,25592,52102,1986-05-24,2011-07-04,9172,dnd2p7,dnd2p7j,dnd2p72,portland,TN,1986,2011,36.58020,-86.51040,36.581667,-86.516389,559.0


In [ ]:
# proximity_score 
# 1000 meters and 730 days (2 years)
# distance and time decay 
# avoids divide by zero error
# generates an appoximate range of possible values 0 - 100 

proximity_df['proximity_score'] = round(
    100 * (
        1 / (1 + proximity_df['distance_meters'] / 5000) *      
        1 / (1 + proximity_df['date_diff_days'] / 365)         
    ), 2)

proximity_df['proximity_score'].describe()

count    286.000000
mean      13.208531
std       14.111548
min        1.730000
25%        4.172500
50%        8.310000
75%       15.562500
max       88.560000
Name: proximity_score, dtype: float64

In [203]:
proximity_df['proximity_rank'] = proximity_df['proximity_score'].rank(method='dense', ascending=False)

proximity_df['proximity_rank'].describe()

count    286.000000
mean     127.405594
std       71.231595
min        1.000000
25%       67.250000
50%      128.500000
75%      191.750000
max      242.000000
Name: proximity_rank, dtype: float64

In [204]:
proximity_df = proximity_df.sort_values('proximity_rank', ascending=True)

proximity_df.head()

,bf_id,uap_id,full_date_bf,full_date_uap,date_diff_days,geohash_6,geohash_7_bf,geohash_7_uap,city,state_code,year_bf,year_uap,latitude_bf,longitude_bf,latitude_uap,longitude_uap,distance_meters,proximity_score,proximity_rank
91,3585,16869,2001-08-01,2001-06-15,47,c2920x,c2920xb,c2920xb,monroe,WA,2001,2001,47.85556,-121.96970,47.855556,-121.969722,2.0,88.56,1.0
186,7815,23666,2003-08-12,2003-10-29,78,9ymb8w,9ymb8wn,9ymb8wn,russellville,AR,2003,2003,35.27833,-93.13361,35.278333,-93.133611,0.0,82.39,2.0
134,4737,17329,2001-05-13,2001-08-03,82,c2293y,c2293yt,c2293yt,chehalis,WA,2001,2001,46.66220,-122.96270,46.662222,-122.962778,6.0,81.56,3.0
207,12587,4941,1988-08-01,1988-06-01,61,dhv65t,dhv65tn,dhv65tk,venice beach,FL,1988,1988,27.09900,-82.45400,27.099722,-82.457778,382.0,79.60,4.0
187,7815,25239,2003-08-12,2004-04-15,247,9ymb8w,9ymb8wn,9ymb8wn,russellville,AR,2003,2004,35.27833,-93.13361,35.278333,-93.133611,0.0,59.64,5.0


In [205]:
# Re-order columns
# IDs, Location, Time, Proximity 
# Sort by Proximity Rank 

ordered_cols = [
    'bf_id', 'uap_id',

    'city', 'state_code',
    'geohash_6', 'geohash_7_bf', 'geohash_7_uap',
    'latitude_bf', 'longitude_bf',
    'latitude_uap', 'longitude_uap',
    
    'full_date_bf', 'full_date_uap', 'date_diff_days',
    'year_bf', 'year_uap',
    
    'distance_meters',
    'proximity_score',
    'proximity_rank'          
]

proximity_df = proximity_df[ordered_cols].sort_values(
    by='proximity_rank', 
    ascending=True 
).reset_index(drop=True)

In [206]:
proximity_df.isnull().sum()

bf_id              0
uap_id             0
city               0
state_code         0
geohash_6          0
geohash_7_bf       0
geohash_7_uap      0
latitude_bf        0
longitude_bf       0
latitude_uap       0
longitude_uap      0
full_date_bf       0
full_date_uap      0
date_diff_days     0
year_bf            0
year_uap           0
distance_meters    0
proximity_score    0
proximity_rank     0
dtype: int64

In [ ]:
proximity_df.to_csv("../data/processed/proximity_v1.csv", index=False)